# What Preprocessing Leakage Cost a Startup Success Model

*8,837 Crunchbase companies. Preprocessing fit inside each fold scores 0.794265, fit on all rows first, 0.794240.*

In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

dataset = "/kaggle/input/big-startup-secsees-fail-dataset-from-crunchbase/big_startup_secsees_dataset.csv"
if not os.path.exists(dataset):
    dataset = "big_startup_secsees_dataset.csv"
df_original = pd.read_csv(dataset)

In [2]:
df_original.head()

,permalink,name,homepage_url,category_list,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,founded_at,first_funding_at,last_funding_at
0,/organization/-fame,#fame,http://livfame.com,Media,10000000,operating,IND,16,Mumbai,Mumbai,1,NaN,2015-01-05,2015-01-05
1,/organization/-qounter,:Qounter,http://www.qounter.com,Application Platforms|Real Time|Social Network...,700000,operating,USA,DE,DE - Other,Delaware City,2,2014-09-04,2014-03-01,2014-10-14
2,/organization/-the-one-of-them-inc-,"(THE) ONE of THEM,Inc.",http://oneofthem.jp,Apps|Games|Mobile,3406878,operating,NaN,NaN,NaN,NaN,1,NaN,2014-01-30,2014-01-30
3,/organization/0-6-com,0-6.com,http://www.0-6.com,Curated Web,2000000,operating,CHN,22,Beijing,Beijing,1,2007-01-01,2008-03-19,2008-03-19
4,/organization/004-technologies,004 Technologies,http://004gmbh.de/en/004-interact,Software,-,operating,USA,IL,"Springfield, Illinois",Champaign,1,2010-01-01,2014-07-24,2014-07-24


Every step below runs on the full dataset, before any train/test split. That is only safe under one rule: a step may run before the split only if it is row-local, meaning the value it produces for a row can be worked out from that row alone. A step that has to read other rows to produce its answer belongs inside the pipeline, where it is refit on the training rows of each fold and never sees held-out data.

The two operations on `funding_total_usd` sit either side of that line. Flagging a row as missing reads one cell of one row, so it returns the same answer on the full dataset as it would on a single row, and `funding_missing` is built here. Filling that missing value with the median reads every row to produce a single number. Computed above the split, that median is measured from the training rows and the held-out rows together, and the test set has quietly shaped the data the model trains on. This is preprocessing leakage: not the model reading a test row, but a value measured from test rows being stitched into the training data.

Dropping companies with no resolved outcome, mapping `status` to a 0/1 target, parsing `-` to `NaN`, subtracting dates, and taking the first entry out of `category_list` are all row-local, so they stay here. Anything that learns a quantity from the data, the median, the scaler's mean and spread, the encoder's category ranking, is held back for the pipeline.

In [3]:
df = df_original[~(df_original['status'] == 'operating')]

In [4]:
df.shape

(13334, 14)

In [5]:
df['success'] = df['status'].map({
    'ipo': 1,
    'acquired': 1,
    'closed': 0
})

In [6]:
df = df.drop(columns=['permalink', 'name', 'homepage_url', 'status', 'state_code', 'region', 'city'])

In [7]:
df.isna().sum()

category_list        1086
funding_total_usd       0
country_code         1991
funding_rounds          0
founded_at           3732
first_funding_at        2
last_funding_at         0
success                 0
dtype: int64

In [8]:
df = df.dropna(subset=['founded_at', 'first_funding_at'])

In [9]:
cols = ['category_list', 'country_code']

for c in cols:
    print(df[c].value_counts(dropna=False).head(10))

category_list
Software               692
Biotechnology          506
NaN                    434
Curated Web            264
Mobile                 207
Enterprise Software    192
E-Commerce             158
Advertising            152
Games                  146
Semiconductors         146
Name: count, dtype: int64
country_code
USA    6275
NaN     958
GBR     394
CAN     289
ISR     171
FRA     144
DEU     138
CHN     120
RUS     112
IND     105
Name: count, dtype: int64


In [10]:
df.dtypes

category_list          str
funding_total_usd      str
country_code           str
funding_rounds       int64
founded_at             str
first_funding_at       str
last_funding_at        str
success              int64
dtype: object

In [11]:
mask = pd.to_numeric(df['funding_total_usd'], errors='coerce').isna()
df.loc[mask, 'funding_total_usd'].value_counts()

funding_total_usd
-    1419
Name: count, dtype: int64

In [12]:
df['funding_total_usd'] = pd.to_numeric(df['funding_total_usd'], errors='coerce')

In [13]:
df['funding_total_usd'].value_counts(dropna=False)

funding_total_usd
NaN            1419
1000000.0       152
500000.0        141
2000000.0       121
100000.0        119
               ... 
3384225.0         1
2257464.0         1
3805520.0         1
866550786.0       1
15419877.0        1
Name: count, Length: 3643, dtype: int64

In [14]:
df['funding_missing'] = df['funding_total_usd'].isna()

In [15]:
dates = ['founded_at', 'first_funding_at', 'last_funding_at']
df[dates] = df[dates].apply(pd.to_datetime, errors='coerce')

In [16]:
df.dtypes

category_list                   str
funding_total_usd           float64
country_code                    str
funding_rounds                int64
founded_at           datetime64[us]
first_funding_at     datetime64[us]
last_funding_at      datetime64[us]
success                       int64
funding_missing                bool
dtype: object

In [17]:
df['days_to_first_funding'] = (df['first_funding_at'] - df['founded_at']).dt.days
df['funding_duration'] = (df['last_funding_at'] - df['first_funding_at']).dt.days

In [18]:
df = df.drop(columns=['founded_at', 'first_funding_at', 'last_funding_at'])

Both new columns come from subtraction, which can produce values the calendar cannot. Nulls, negatives and zeros each need checking, and a negative and a zero turn out to mean very different things.

In [19]:
days = ['days_to_first_funding', 'funding_duration']

summary = pd.DataFrame({
    'null': df[days].isna().sum(),
    'negative': (df[days] < 0).sum(),
    'zero': (df[days] == 0).sum()
})

display(summary)
print(f"Companies with a single funding round: {(df['funding_rounds'] == 1).sum()}")

,null,negative,zero
days_to_first_funding,0,763,663
funding_duration,0,0,5311


Companies with a single funding round: 5280


A negative `days_to_first_funding` records funding arriving before the company existed, which cannot happen, so those 763 rows are broken records and come out. A zero `funding_duration` is a different kind of finding. It is not an impossible value: it describes a company whose first and last funding fell on the same day. Dropping those would cut more than half the remaining rows and, worse, would remove a real category of company rather than a mistake, so they stay.

5,311 companies show a `funding_duration` of zero, but only 5,280 raised a single round. A zero duration normally means one round and nothing after, so those two counts should describe the same companies, and 31 of them do not. That leaves one explanation worth testing: a company that raised more than one round on the same day, and never raised again.

In [20]:
mask = (df['funding_duration'] == 0) & (df['funding_rounds'] > 1)
comparison = df.loc[mask, ['funding_duration', 'funding_rounds']]
display(comparison)
print(f'Companies with funding duration "zero" but more than one funding round: {mask.sum()}')

,funding_duration,funding_rounds
133,0,2
1689,0,2
2825,0,2
4128,0,2
4917,0,2
5968,0,2
7258,0,2
8345,0,2
11246,0,2
11411,0,2


Companies with funding duration "zero" but more than one funding round: 31


All 31 raised every round they ever raised on the same day. Together with the 5,280 single-round companies, they account for all 5,311 zero-duration rows, so the gap is fully explained and no third kind of company is left to look for.

In [21]:
df = df[df['days_to_first_funding'] >= 0]

In [39]:
# class proportions
(df['success'].value_counts(normalize=True)*100).round(1)

success
1    58.9
0    41.1
Name: proportion, dtype: float64

In [40]:
# filling the unfinished company's funding_duration as 0 will make the model push it towards failure (target leakage)
print(f"Success rate of companies where `funding_duration` is zero: {df.loc[df['funding_duration'] == 0, 'success'].mean():.2f}")

Success rate of companies where `funding_duration` is zero: 0.46


### Here

This comes back to the decision to leave out still operating companies earlier in the notebook since given the nature of the data (i.e dataset has both finished and unfinished stories), there is a target leakage. Had the still operating companies been chosen to include in the model and if the model were to train to predict their outcome, their `funding_duration` rows will be target leakage since we don't know their full stories yet. If they have so far only raised one funding round or multiple funding rounds on the same day, their `funding_duration` will be zero which the model has 54% more chance to predict them as failure, therefore creating target leakage.

In [23]:
df['category'] = df['category_list'].str.split('|').str[0]
df = df.drop(columns='category_list')

In [24]:
print(f"Category (Null): {df['category'].isna().sum()}")

Category (Null): 407


In [25]:
X = df.drop(columns='success')
y = df['success']

In order to have the fold data to have the same proportion as the test data, stratify=y is used.

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [28]:
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")

X_train: (7069, 7)
y_train: (7069,)
X_test: (1768, 7)
y_test: (1768,)


In [29]:
print("funding_total_usd")
print(f"{'Mean:':<7}{df['funding_total_usd'].mean():>10,.0f}")
print(f"{'Median:':<7}{df['funding_total_usd'].median():>10,.0f}")

funding_total_usd
Mean:  39,844,438
Median: 6,000,000


`funding_total_usd` has NaN rows and needs to be filled with either mean or median. Median makes more sense here since mean is 6 times larger than median, aka heavily right skewed, and it will inflate the rows with missing values. To avoid data leakage, median filling process is put into hte pipeline where it is computed only from train data.

For one hot encoding, maximum categories is set at 11 the most frequent 10 plus infrequent column if a category that wasn't in the training data appears.

In [30]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(max_categories=11, handle_unknown='infrequent_if_exist'))
])

In [31]:
ct = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, ['funding_total_usd', 'funding_rounds', 'days_to_first_funding', 'funding_duration', 'funding_missing']),
        ('cat', categorical_transformer, ['country_code', 'category'])
    ])

In [32]:
pipeline = Pipeline(steps=[
    ('prep', ct),
    ('model', LogisticRegression())
])

In [33]:
score = cross_val_score(
    pipeline,
    X_train, y_train,
    cv=5,
    scoring='roc_auc'
)
print(score)
print(f"Mean 5-fold CV ROC-AUC: {score.mean():.6f}")

[0.79327607 0.79268482 0.7979815  0.78604695 0.80133812]
Mean 5-fold CV ROC-AUC: 0.794265


## Measuring the leak

In [34]:
X_leaky = ct.fit_transform(X)

In [35]:
X_leaky_train, X_leaky_test, y_leaky_train, y_leaky_test = train_test_split(
    X_leaky, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [36]:
leaky_score = cross_val_score(
    LogisticRegression(),
    X_leaky_train, y_leaky_train,
    cv=5,
    scoring='roc_auc'
)
print(leaky_score)
print(f"Mean 5-fold CV ROC-AUC: {leaky_score.mean():.6f}")

[0.79312929 0.79312309 0.79853318 0.78477415 0.80164031]
Mean 5-fold CV ROC-AUC: 0.794240


In [37]:
print(leaky_score - score)

[-0.00014678  0.00043827  0.00055168 -0.0012728   0.00030219]


In conclusion, there is barely any differnce on the AUC score leaking the data vs. using a pipeline. 